In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 8


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.035590909421444
Epoch 2/100, Loss: 2.038944721221924
Epoch 3/100, Loss: 1.9562358856201172
Epoch 4/100, Loss: 2.048598773777485
Epoch 5/100, Loss: 2.0083866007626057
Epoch 6/100, Loss: 2.692954808473587
Epoch 7/100, Loss: 2.0466955676674843
Epoch 8/100, Loss: 2.021146610379219
Epoch 9/100, Loss: 1.9365567862987518
Epoch 10/100, Loss: 1.9462097063660622
Epoch 11/100, Loss: 1.9640254229307175
Epoch 12/100, Loss: 2.029901944100857
Epoch 13/100, Loss: 1.7801090329885483
Epoch 14/100, Loss: 1.878258928656578
Epoch 15/100, Loss: 2.037112422287464
Epoch 16/100, Loss: 1.9873804822564125


Epoch 17/100, Loss: 2.087942972779274
Epoch 18/100, Loss: 2.028234124183655
Epoch 19/100, Loss: 2.0892146602272987
Epoch 20/100, Loss: 1.9030761495232582
Epoch 21/100, Loss: 1.950657095760107
Epoch 22/100, Loss: 1.8684012219309807
Epoch 23/100, Loss: 1.9059557989239693
Epoch 24/100, Loss: 2.050698347389698
Epoch 25/100, Loss: 2.02660808339715
Epoch 26/100, Loss: 2.0105938240885735
Epoch 27/100, Loss: 1.8917210027575493
Epoch 28/100, Loss: 2.0764621198177338
Epoch 29/100, Loss: 2.1205920167267323
Epoch 30/100, Loss: 1.8832296431064606
Epoch 31/100, Loss: 2.0839821323752403
Epoch 32/100, Loss: 1.952197503298521


Epoch 33/100, Loss: 2.1421943083405495
Epoch 34/100, Loss: 2.07781945168972
Epoch 35/100, Loss: 1.974451258778572
Epoch 36/100, Loss: 1.931841529905796
Epoch 37/100, Loss: 1.8454245440661907
Epoch 38/100, Loss: 2.038918759673834
Epoch 39/100, Loss: 1.9542393386363983
Epoch 40/100, Loss: 1.8785130232572556
Epoch 41/100, Loss: 1.9968030750751495
Epoch 42/100, Loss: 1.7827986776828766
Epoch 43/100, Loss: 2.511078178882599
Epoch 44/100, Loss: 1.8793429136276245
Epoch 45/100, Loss: 1.8408079072833061
Epoch 46/100, Loss: 2.5177888944745064


Epoch 47/100, Loss: 2.026148594915867
Epoch 48/100, Loss: 2.044952653348446
Epoch 49/100, Loss: 2.0708787962794304
Epoch 50/100, Loss: 2.1373640447854996
Epoch 51/100, Loss: 2.1401063799858093
Epoch 52/100, Loss: 1.8039430603384972
Epoch 53/100, Loss: 1.9509591832756996
Epoch 54/100, Loss: 2.068114586174488
Epoch 55/100, Loss: 2.1389994472265244
Epoch 56/100, Loss: 1.9984481558203697
Epoch 57/100, Loss: 1.974209651350975
Epoch 58/100, Loss: 1.9878541678190231


Epoch 59/100, Loss: 2.0264325961470604
Epoch 60/100, Loss: 1.830891266465187
Epoch 61/100, Loss: 1.8077306896448135
Epoch 62/100, Loss: 1.8558809384703636
Epoch 63/100, Loss: 1.9782431051135063
Epoch 64/100, Loss: 2.0259906724095345
Epoch 65/100, Loss: 1.9748735427856445
Epoch 66/100, Loss: 1.8983824700117111
Epoch 67/100, Loss: 1.9890545904636383
Epoch 68/100, Loss: 1.9332358613610268
Epoch 69/100, Loss: 1.9606510028243065
Epoch 70/100, Loss: 1.8933763056993484
Epoch 71/100, Loss: 2.220673307776451
Epoch 72/100, Loss: 2.097820498049259
Epoch 73/100, Loss: 2.0060611814260483


...

Epoch 24/100, Loss: 2.3063539266586304
Epoch 25/100, Loss: 2.294897384941578
Epoch 26/100, Loss: 2.3735892549157143
Epoch 27/100, Loss: 2.2660916596651077
Epoch 28/100, Loss: 2.399079903960228
Epoch 29/100, Loss: 2.267690859735012
Epoch 30/100, Loss: 2.459335871040821
Epoch 31/100, Loss: 2.3566762506961823
Epoch 32/100, Loss: 2.343326300382614
Epoch 33/100, Loss: 2.0943276807665825
Epoch 34/100, Loss: 2.1681156754493713
Epoch 35/100, Loss: 2.22190473228693
Epoch 36/100, Loss: 2.065957024693489
Epoch 37/100, Loss: 2.110607512295246
Epoch 38/100, Loss: 2.1557332053780556
Epoch 39/100, Loss: 2.2892086803913116
Epoch 40/100, Loss: 2.18908528983593


Epoch 41/100, Loss: 2.308377258479595
Epoch 42/100, Loss: 2.439729429781437
Epoch 43/100, Loss: 2.4535649716854095
Epoch 44/100, Loss: 2.104226239025593
Epoch 45/100, Loss: 1.9645323753356934
Epoch 46/100, Loss: 2.2162512689828873
Epoch 47/100, Loss: 2.2816765159368515
Epoch 48/100, Loss: 2.177124708890915
Epoch 49/100, Loss: 2.4936850890517235
Epoch 50/100, Loss: 2.4189577028155327
Epoch 51/100, Loss: 2.256023198366165
Epoch 52/100, Loss: 2.482049509882927
Epoch 53/100, Loss: 2.1862968653440475
Epoch 54/100, Loss: 2.1736282482743263
Epoch 55/100, Loss: 2.265537090599537
Epoch 56/100, Loss: 1.960379771888256
Epoch 57/100, Loss: 2.364606484770775
Epoch 58/100, Loss: 2.202732138335705


Epoch 59/100, Loss: 2.211122378706932
Epoch 60/100, Loss: 2.4874259755015373
Epoch 61/100, Loss: 2.0714286640286446
Epoch 62/100, Loss: 2.095571029931307
Epoch 63/100, Loss: 2.2666308656334877
Epoch 64/100, Loss: 2.3340419307351112
Epoch 65/100, Loss: 2.3695829287171364
Epoch 66/100, Loss: 2.434673562645912
Epoch 67/100, Loss: 2.4114246517419815
Epoch 68/100, Loss: 2.0692432895302773
Epoch 69/100, Loss: 2.3084555491805077
Epoch 70/100, Loss: 2.2604740485548973
Epoch 71/100, Loss: 2.4218231216073036
Epoch 72/100, Loss: 2.35936526209116


Epoch 73/100, Loss: 2.216894492506981
Epoch 74/100, Loss: 2.1559871211647987
Epoch 75/100, Loss: 2.3359364569187164
Epoch 76/100, Loss: 2.1161911860108376
Epoch 77/100, Loss: 2.1582909524440765
Epoch 78/100, Loss: 2.4130637645721436
Epoch 79/100, Loss: 2.3991457670927048
Epoch 80/100, Loss: 2.3202975392341614
Epoch 81/100, Loss: 2.2830350026488304
Epoch 82/100, Loss: 2.392579033970833
Epoch 83/100, Loss: 2.2696264907717705
Epoch 84/100, Loss: 2.330961287021637
Epoch 85/100, Loss: 2.1996250450611115
Epoch 86/100, Loss: 2.32193610817194
Epoch 87/100, Loss: 2.440689541399479
Epoch 88/100, Loss: 2.407063849270344
Epoch 89/100, Loss: 2.1940606385469437
Epoch 90/100, Loss: 2.4633685871958733


Epoch 91/100, Loss: 2.490214169025421
Epoch 92/100, Loss: 2.3499632477760315
Epoch 93/100, Loss: 2.1541339829564095
Epoch 94/100, Loss: 2.2994924634695053
Epoch 95/100, Loss: 2.156476490199566
Epoch 96/100, Loss: 2.282227136194706
Epoch 97/100, Loss: 2.3149693980813026
Epoch 98/100, Loss: 2.1778949350118637
Epoch 99/100, Loss: 2.291918210685253
Epoch 100/100, Loss: 2.3710713982582092
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.7265
